# Exploratory Data Analysis — Hotel Bookings

Quick look at the dataset that feeds the cancellation model: shape, missingness, target balance, and how cancellation varies across key features.

Run from the repo root (`pip install -e .` first, or open via `notebooks/colab_quickstart.ipynb` in Colab).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from hotel_cancel.config import Config
from hotel_cancel.data import load_raw
from hotel_cancel.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET

config = Config.load()
df = load_raw(config.raw_data_path)
print(df.shape)
df.head()

## Schema, dtypes, and missing values (used columns only)

In [ ]:
used = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]
summary = pd.DataFrame({
    "dtype": df[used].dtypes.astype(str),
    "nulls": df[used].isnull().sum(),
    "n_unique": df[used].nunique(),
})
summary

## Target balance

In [ ]:
counts = df[TARGET].value_counts().sort_index()
print(counts)
print("cancellation rate:", round(df[TARGET].mean(), 3))
counts.plot.bar(title="is_canceled (0 = kept, 1 = cancelled)")
plt.tight_layout(); plt.show()

## Cancellation rate by categorical feature

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), CATEGORICAL_FEATURES):
    df.groupby(col)[TARGET].mean().sort_values().plot.barh(ax=ax)
    ax.set_title(f"Cancellation rate by {col}")
    ax.set_xlabel("P(cancel)")
plt.tight_layout(); plt.show()

## Lead time vs cancellation
Longer lead times tend to cancel more often.

In [ ]:
df.boxplot(column="lead_time", by=TARGET, grid=False)
plt.suptitle(""); plt.title("Lead time by cancellation")
plt.tight_layout(); plt.show()

## Numeric correlations with the target

In [ ]:
corr = df[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
corr.plot.barh(title="Correlation with is_canceled")
plt.tight_layout(); plt.show()
corr